# 04 — Merge segments and export to Excel

Two operations the API performs after the classifier+analyzer call:

1. **`sec_service.merge_segments(raw)`** — flatten the per-category segments
   into a single CU payload (`contents[0].fields.financialTables`), keeping
   classifier order and stamping each table with its `_segmentCategory` +
   page range.
2. **`sec_service.export_to_excel(merged, out_path)`** — render a multi-sheet
   `.xlsx` with hierarchy-aware indentation, bold section headers, and
   top-bordered subtotals. This is the same file the **Download Excel**
   button on the SEC page serves.


In [ ]:
from _lib import sec_service, sec_excel
import json

sample = sec_service.list_samples()[0]['file_name']
# Use a cached payload to avoid Azure on every notebook run.
cached = sec_service.load_cached_payload(sample)
if cached is None:
    sec_service.ensure_analyzers()
    raw, _ = sec_service.classify_and_extract((sec_service.ATTACH_DIR / sample).read_bytes())
    merged = sec_service.merge_segments(raw)
else:
    merged = cached
print('tables in merged payload:', len(merged['contents'][0]['fields']['financialTables']['valueArray']))

## Export the merged payload to Excel

In [ ]:
from pathlib import Path
out_dir = sec_service.SEC_DEMO_ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{Path(sample).stem}.xlsx"
sec_service.export_to_excel(merged, out_path)
print('wrote', out_path, f"({out_path.stat().st_size/1024:.1f} KB)")

## Normalize the same payload for the UI

The SEC page renders these normalized tables (`statement_type`, `period_headers`, hierarchical `rows[]`).

In [ ]:
normalized = sec_excel.load_from_payload(merged)
for t in normalized:
    print(f"  {t['statementType']:22s}  rows={len(t['rows']):3d}  cols={len(t['periodHeaders'])}  '{t['tableTitle'][:60]}'")